# Notebook 18: two encoders inside one TabPFN, not two arms

The blend is finished. An oracle fitted directly on the holdout over all eleven arms
scores 0.5862 against our shipped 0.5891, so perfect-hindsight reweighting is worth
0.0029 and nothing we add as a seventh arm can matter.

A CPU screen yesterday found that concatenating two encoders **inside one model** beat
either alone by +0.0074 dev under gradient boosting, and that three sources were worse
than one. Our two best arms are the TabPFN versions of exactly those two single sources:
`tab` is TabPFN over chemprop_medium plus the qHTS block, `n15` is TabPFN over
surrogate_adme plus the same block. The two-source version has never been run.

So this notebook runs **both** configurations end to end, in the same notebook, on the
same folds, with the same code:

| tag | sources |
| --- | --- |
| `s1` | surrogate_adme only, which should reproduce `n15` at dev macro ~0.6370 |
| `s2` | chemprop_medium + surrogate_adme |

`s1` is the control. If it does not land near 0.637 then something differs from the n15
run and the `s2` number cannot be interpreted, so check that line first.

Both encoders are pre-reduced to 200 components in the payload. The PCA was fitted on
train and test features together, which is unsupervised and uses no labels. This matches
how `n15` was built, so the control is a fair one.

**Runtime** about 80 minutes, two configurations at roughly 40 each. Every stage
checkpoints to Drive; Run-all resumes after a disconnect.


In [ ]:
import subprocess
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout or 'NO GPU: switch the runtime to a GPU before going on')

In [ ]:
PAYLOAD_FILE = '/content/drive/MyDrive/cyp_payloads/twoenc_7c64432959.tgz'

import os, tarfile
os.makedirs('/content/cache', exist_ok=True)
from google.colab import drive
drive.mount('/content/drive')
with tarfile.open(PAYLOAD_FILE) as t:
    t.extractall('/content/cache')
OUTDIR = '/content/drive/MyDrive/cyp_results'
os.makedirs(OUTDIR, exist_ok=True)
assert os.path.ismount('/content/drive'), 'Drive is not mounted; stop here, the run could not resume without it'
print('cache:', sorted(os.listdir('/content/cache')))
print('writing to', OUTDIR)


In [ ]:
!pip -q install tabpfn==2.2.1


## Both configurations, same folds, same code

`s1` is the control and must land near dev macro 0.6370. Read that line before reading anything else.


In [ ]:
import json, os, time, numpy as np, pandas as pd, torch
from pathlib import Path
from scipy.stats import spearmanr
from sklearn.model_selection import GroupKFold
from sklearn.impute import SimpleImputer
from tabpfn import TabPFNRegressor

CACHE = Path('/content/cache'); OUT = Path(OUTDIR)
ISO = ['CYP1A2', 'CYP2C9', 'CYP2D6', 'CYP3A4']
REF_TAB = {'CYP1A2': 0.5610, 'CYP2C9': 0.7189, 'CYP2D6': 0.4219, 'CYP3A4': 0.8074}
REF_N15 = {'CYP1A2': 0.5639, 'CYP2C9': 0.7268, 'CYP2D6': 0.4460, 'CYP3A4': 0.8115}
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

meta = pd.read_csv(CACHE / 'meta_train.csv.gz')
hold = np.load(CACHE / 'holdout_mask.npy')
G = np.load(CACHE / 'scaffold_groups.npy', allow_pickle=True)
SRC = {
    'medium':    (np.load(CACHE / 'emb_medium200_train.npy'), np.load(CACHE / 'emb_medium200_test.npy')),
    'surrogate': (np.load(CACHE / 'emb_surr200_train.npy'),   np.load(CACHE / 'emb_surr200_test.npy')),
}
RAW = np.hstack([np.load(CACHE / 'qhts_tr.npy'), np.load(CACHE / 'aux_train.npy')]).astype(np.float32)
RAW_TE = np.hstack([np.load(CACHE / 'qhts_te.npy'), np.load(CACHE / 'aux_test.npy')]).astype(np.float32)
print('device', DEVICE, '| raw qhts+aux', RAW.shape[1], 'cols')
for k, (a, b) in SRC.items():
    print(f'  {k}: train {a.shape} test {b.shape}')
    assert len(a) == len(meta) and len(b) == len(RAW_TE)
assert len(RAW) == len(meta)

CFG = {'s1': ['surrogate'], 's2': ['medium', 'surrogate']}
KW = dict(device=DEVICE, random_state=42, ignore_pretraining_limits=True)

def feats(names, tr_idx, ev_rows, ev_is_test):
    """Impute on training rows only; the embeddings are already reduced."""
    tr_blocks, ev_blocks = [], []
    for n in names:
        a, b = SRC[n]
        tr_blocks.append(a[tr_idx])
        ev_blocks.append(b[ev_rows] if ev_is_test else a[ev_rows])
    r_tr = RAW[tr_idx]
    r_ev = RAW_TE[ev_rows] if ev_is_test else RAW[ev_rows]
    ri = SimpleImputer(strategy='median').fit(r_tr)
    tr_blocks.append(ri.transform(r_tr)); ev_blocks.append(ri.transform(r_ev))
    return (np.hstack(tr_blocks).astype(np.float32),
            np.hstack(ev_blocks).astype(np.float32))

def fit_predict(names, tr_idx, ev_rows, y, ev_is_test=False):
    Xt, Xe = feats(names, tr_idx, ev_rows, ev_is_test)
    return TabPFNRegressor(**KW).fit(Xt, y[tr_idx]).predict(Xe)

# smoke test: one small fit per configuration, shapes printed, fails before the long run
y0 = meta['CYP3A4_pIC50_direct_inhibition'].to_numpy(float)
ok = np.flatnonzero(np.isfinite(y0))[:90]
for tag, names in CFG.items():
    Xt, Xe = feats(names, ok[:70], ok[70:], False)
    p = TabPFNRegressor(**KW).fit(Xt, y0[ok[:70]]).predict(Xe)
    print(f'smoke {tag} ({"+".join(names)}): train {Xt.shape} eval {Xe.shape} '
          f'finite {bool(np.isfinite(p).all())}')
print('smoke test passed\n')

def save_atomic(path, **kw):
    tmp = Path(str(path) + '.tmp.npz'); np.savez_compressed(tmp, **kw); os.replace(tmp, path)

ALL = []
t_all = time.time()
for tag, names in CFG.items():
    print(f'=== {tag}: {"+".join(names)} ===', flush=True)
    for iso in ISO:
        final = OUT / f'{tag}_{iso}.npz'
        y = meta[f'{iso}_pIC50_direct_inhibition'].to_numpy(float)
        obs = np.isfinite(y)
        dev = np.flatnonzero(obs & ~hold); ho = np.flatnonzero(obs & hold); allr = np.flatnonzero(obs)
        if final.exists():
            z = np.load(final); oof, hp = z['oof'], z['hold']
            print(f'{iso}: done earlier, skipping', flush=True)
        else:
            part = OUT / f'part_{tag}_{iso}.npz'
            oof = np.full(len(dev), np.nan); hp = np.full(len(ho), np.nan)
            tp = np.full(len(RAW_TE), np.nan); done = []
            if part.exists():
                z = np.load(part, allow_pickle=True)
                oof, hp, tp = z['oof'], z['hp'], z['tp']; done = list(z['done'])
                print(f'{iso}: resuming, stages finished {done}', flush=True)
            def ckpt(): save_atomic(part, oof=oof, hp=hp, tp=tp, done=np.array(done))
            t0 = time.time()
            for k, (tr, va) in enumerate(GroupKFold(5).split(dev, dev, G[dev])):
                if f'f{k}' in done: continue
                oof[va] = fit_predict(names, dev[tr], dev[va], y)
                done.append(f'f{k}'); ckpt()
                print(f'  {iso} fold {k} ({time.time()-t0:.0f}s)', flush=True)
            if 'hold' not in done:
                hp = fit_predict(names, dev, ho, y); done.append('hold'); ckpt()
                print(f'  {iso} holdout ({time.time()-t0:.0f}s)', flush=True)
            if 'test' not in done:
                tp = fit_predict(names, allr, np.arange(len(RAW_TE)), y, ev_is_test=True)
                done.append('test'); ckpt()
                print(f'  {iso} test ({time.time()-t0:.0f}s)', flush=True)
            assert np.isfinite(oof).all() and np.isfinite(hp).all() and np.isfinite(tp).all()
            save_atomic(final, oof=oof, dev=dev, hold=hp, ho=ho, test=tp)
            part.unlink(missing_ok=True)
        d = float(spearmanr(y[dev], oof).statistic); h = float(spearmanr(y[ho], hp).statistic)
        ALL.append(dict(tag=tag, iso=iso, dev_spearman=d, holdout_spearman=h,
                        ref_tab=REF_TAB[iso], ref_n15=REF_N15[iso]))
        json.dump(ALL, open(OUT / 'results_nb18.json', 'w'), indent=2)
        print(f'{iso}: dev {d:.4f} | holdout {h:.4f}  (tab {REF_TAB[iso]:.4f}, '
              f'n15 {REF_N15[iso]:.4f})', flush=True)

r = pd.DataFrame(ALL)
print('\n' + '=' * 62)
for tag in CFG:
    s = r[r.tag == tag]
    if len(s): print(f'{tag} ({"+".join(CFG[tag])}): macro dev {s.dev_spearman.mean():.4f} '
                     f'| macro holdout {s.holdout_spearman.mean():.4f}')
print(f'control target: s1 should land near dev macro 0.6370 (that is n15)')
print(f'tab macro dev 0.6273')
if len(r[r.tag == 's1']) == 4 and len(r[r.tag == 's2']) == 4:
    g = r[r.tag == 's2'].dev_spearman.mean() - r[r.tag == 's1'].dev_spearman.mean()
    print(f'\ntwo-source gain over one: {g:+.4f} dev '
          f'(the CPU screen predicted about +0.0074)')
print(f'total {(time.time()-t_all)/60:.1f} min')


In [ ]:
import shutil
shutil.make_archive('/content/nb18_results', 'zip', OUT)
try:
    from google.colab import files
    files.download('/content/nb18_results.zip')
except Exception as e:
    print('download skipped:', type(e).__name__, '- results are on Drive in', OUT)
